In [1]:
import warnings
warnings.filterwarnings('ignore')

import argparse
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)


# ==============================
# 1. Cấu hình
# ==============================
RANDOM_STATE = 42
DATA_PATH = Path(r'C:\Users\Admin\Downloads\annotated_clauses_topics.csv')
ARTIFACT_DIR = Path('artifacts_linear_svm')

TEXT_COL = 'Clause'
TARGET_COL = 'code'


# ==============================
# 2. Hàm tiện ích
# ==============================

def print_header(title: str):
    print('\n' + '=' * 100)
    print(title)
    print('=' * 100)


def basic_eda(df: pd.DataFrame):
    print_header('EDA TÓM TẮT')
    print(f'Shape dữ liệu: {df.shape}')
    print('\nCác cột:', list(df.columns))
    print('\nSố lượng missing value theo cột:')
    print(df.isnull().sum())
    print(f'\nSố dòng duplicated hoàn toàn: {df.duplicated().sum()}')
    print('\nPhân bố lớp (code):')
    print(df[TARGET_COL].value_counts())
    print('\nTỷ lệ lớp (%):')
    print((df[TARGET_COL].value_counts(normalize=True) * 100).round(2))
    print('\nĐộ dài Clause (ký tự) mô tả nhanh:')
    clause_len = df[TEXT_COL].astype(str).str.len()
    print(clause_len.describe().round(2))


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Loại bỏ missing ở cột đầu vào/đầu ra nếu có
    df = df.dropna(subset=[TEXT_COL, TARGET_COL])

    # Chuẩn hoá dữ liệu text ở mức tối thiểu
    df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
    df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip()

    # Loại bỏ các dòng Clause rỗng sau khi strip
    df = df[df[TEXT_COL] != ''].copy()

    # Loại bỏ trùng theo cặp (Clause, code) để giảm nhiễu / leakage nhẹ
    before = len(df)
    df = df.drop_duplicates(subset=[TEXT_COL, TARGET_COL]).reset_index(drop=True)
    after = len(df)

    print_header('LÀM SẠCH DỮ LIỆU')
    print(f'Số dòng trước làm sạch: {before}')
    print(f'Số dòng sau làm sạch:   {after}')
    print(f'Số dòng bị loại:        {before - after}')

    return df


def split_data(df: pd.DataFrame):
    X = df[TEXT_COL]
    y = df[TARGET_COL]

    # Bước 1: tách test 10%
    X_temp, X_test, y_temp, y_test = train_test_split(
        X,
        y,
        test_size=0.10,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    # Bước 2: từ 90% còn lại tách valid sao cho cuối cùng là 80/10/10
    # 10/90 = 1/9
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_temp,
        y_temp,
        test_size=1/9,
        stratify=y_temp,
        random_state=RANDOM_STATE,
    )

    print_header('KÍCH THƯỚC TẬP TRAIN / VALID / TEST')
    print(f'Train: {len(X_train)} ({len(X_train)/len(df):.2%})')
    print(f'Valid: {len(X_valid)} ({len(X_valid)/len(df):.2%})')
    print(f'Test:  {len(X_test)} ({len(X_test)/len(df):.2%})')

    return X_train, X_valid, X_test, y_train, y_valid, y_test


from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_selection import SelectKBest, chi2

CUSTOM_STOPWORDS = {
    'good', 'nice', 'great', 'beautiful', 'amazing', 'wonderful',
    'worth', 'best', 'bad', 'okay', 'ok', 'interesting',
    'place', 'experience', 'visit', 'visiting'
}

def build_pipeline() -> Pipeline:
    """Pipeline SVM cơ bản. Các tham số chính sẽ được tune bằng RandomizedSearchCV."""
    stop_words = list(set(ENGLISH_STOP_WORDS).union(CUSTOM_STOPWORDS))

    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            strip_accents='unicode',
            lowercase=True,
            analyzer='word',
            stop_words=stop_words,
        )),
        ('select', SelectKBest(score_func=chi2)),
        ('clf', LinearSVC(
            random_state=RANDOM_STATE,
            max_iter=30000,
        )),
    ])
    return pipeline


def get_param_distributions():
    """Không gian hyperparameter để tuning cho TF-IDF + feature selection + LinearSVC."""
    return {
        # TF-IDF hyperparameters
        'tfidf__ngram_range': [(1, 1), (1, 2)],
        'tfidf__min_df': [2, 3, 5],
        'tfidf__max_df': [0.80, 0.85, 0.90, 0.95],
        'tfidf__max_features': [5000, 7000, 10000, 15000],
        'tfidf__sublinear_tf': [True, False],
        'tfidf__norm': ['l2'],

        # Chi-square feature selection
        'select__k': [1000, 2000, 3000, 5000, 'all'],

        # Linear SVM hyperparameters
        'clf__C': [0.01, 0.03, 0.05, 0.1, 0.3, 0.5, 1.0],
        'clf__class_weight': [None, 'balanced'],
        'clf__loss': ['squared_hinge'],
    }


def tune_hyperparameters(X_train, y_train, n_iter=20, n_jobs=1):
    """Tune hyperparameters bằng RandomizedSearchCV trên tập train."""
    print_header('HYPERPARAMETER TUNING - RANDOMIZED SEARCH CV')

    pipeline = build_pipeline()
    param_distributions = get_param_distributions()

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring='f1_macro',
        cv=cv,
        n_jobs=n_jobs,
        verbose=2,
        random_state=RANDOM_STATE,
        refit=True,
        return_train_score=True,
    )

    search.fit(X_train, y_train)

    print('\nBest CV macro-F1:', round(search.best_score_, 4))
    print('\nBest hyperparameters:')
    for key, value in search.best_params_.items():
        print(f'  {key}: {value}')

    tuning_results = pd.DataFrame(search.cv_results_).sort_values(
        by='rank_test_score'
    )
    tuning_results_path = ARTIFACT_DIR / 'svm_randomized_search_cv_results.csv'
    tuning_results.to_csv(tuning_results_path, index=False)
    print(f'\nĐã lưu kết quả tuning tại: {tuning_results_path.resolve()}')

    return search, tuning_results


def evaluate_split(model, X, y_true, split_name: str, labels):
    y_pred = model.predict(X)

    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=labels,
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report_dict).transpose()

    # Thêm cột confusion matrix để phân tích sâu hơn nếu cần
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_df = pd.DataFrame(cm, index=[f'true_{l}' for l in labels], columns=[f'pred_{l}' for l in labels])

    print_header(f'ĐÁNH GIÁ TRÊN TẬP {split_name.upper()}')
    print(f'Accuracy           : {acc:.4f}')
    print(f'Precision (macro)  : {p_macro:.4f}')
    print(f'Recall (macro)     : {r_macro:.4f}')
    print(f'F1-score (macro)   : {f1_macro:.4f}')
    print(f'Precision(weighted): {p_weighted:.4f}')
    print(f'Recall (weighted)  : {r_weighted:.4f}')
    print(f'F1-score (weighted): {f1_weighted:.4f}')
    print('\nClassification report:')
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0, digits=4))

    report_path = ARTIFACT_DIR / f'classification_report_{split_name.lower()}.csv'
    cm_path = ARTIFACT_DIR / f'confusion_matrix_{split_name.lower()}.csv'
    report_df.to_csv(report_path)
    cm_df.to_csv(cm_path)

    metrics = {
        'split': split_name,
        'accuracy': acc,
        'precision_macro': p_macro,
        'recall_macro': r_macro,
        'f1_macro': f1_macro,
        'precision_weighted': p_weighted,
        'recall_weighted': r_weighted,
        'f1_weighted': f1_weighted,
    }
    return metrics, report_df, cm_df


def compare_overfitting(train_metrics, valid_metrics, test_metrics, cv_best_score=None):
    print_header('KIỂM TRA DẤU HIỆU OVERFITTING')
    print(f"Train macro-F1     : {train_metrics['f1_macro']:.4f}")
    print(f"Valid macro-F1     : {valid_metrics['f1_macro']:.4f}")
    print(f"Test macro-F1      : {test_metrics['f1_macro']:.4f}")

    gap_train_valid = train_metrics['f1_macro'] - valid_metrics['f1_macro']
    gap_train_test = train_metrics['f1_macro'] - test_metrics['f1_macro']

    print(f"Gap train-valid    : {gap_train_valid:.4f}")
    print(f"Gap train-test     : {gap_train_test:.4f}")

    if gap_train_valid > 0.10:
        print('\nCảnh báo: vẫn còn overfitting.')
    elif gap_train_valid > 0.05:
        print('\nCó overfitting nhẹ.')
    else:
        print('\nMức chênh ổn.')

def save_model(model, label_order, best_params):
    model_path = ARTIFACT_DIR / 'linear_svm_topic_classifier.joblib'
    meta_path = ARTIFACT_DIR / 'linear_svm_topic_classifier_meta.json'

    joblib.dump(model, model_path)

    metadata = {
        'model_type': 'Pipeline(TfidfVectorizer + LinearSVC)',
        'text_column': TEXT_COL,
        'target_column': TARGET_COL,
        'label_order': list(label_order),
        'best_params': best_params,
        'random_state': RANDOM_STATE,
    }

    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2, default=str)

    print_header('LƯU MODEL')
    print(f'Đã lưu model tại: {model_path.resolve()}')
    print(f'Đã lưu metadata tại: {meta_path.resolve()}')


def save_inference_example():
    inference_code = r'''import joblib
import pandas as pd

# Load model đã lưu
model = joblib.load('artifacts_linear_svm/linear_svm_topic_classifier.joblib')

# Dự đoán cho 1 câu mới
new_clauses = [
    'The beach is clean and beautiful',
    'Staff were rude and not helpful',
]

preds = model.predict(new_clauses)

result = pd.DataFrame({
    'Clause': new_clauses,
    'Predicted_code': preds,
})
print(result)
'''
    path = ARTIFACT_DIR / 'load_and_predict_example.py'
    path.write_text(inference_code, encoding='utf-8')
    print(f'\nĐã lưu ví dụ load model và predict tại: {path.resolve()}')


# ==============================
# 3. Chạy pipeline chính
# ==============================
if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--data_path', default=str(DATA_PATH))
    parser.add_argument('--artifact_dir', default=str(ARTIFACT_DIR))
    parser.add_argument('--n_iter', type=int, default=20)
    parser.add_argument('--n_jobs', type=int, default=1)

    # Dùng parse_known_args để bỏ qua các tham số nội bộ mà Jupyter tự truyền vào
    args, _ = parser.parse_known_args()

    DATA_PATH = Path(args.data_path)
    ARTIFACT_DIR = Path(args.artifact_dir)
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

    if not DATA_PATH.exists():
        raise FileNotFoundError(
            f'Không tìm thấy file dữ liệu: {DATA_PATH.resolve()}\n'
            'Hãy đặt file sample_set_Labeled.csv cùng thư mục với notebook hoặc sửa DATA_PATH.'
        )

    # Đọc dữ liệu
    df = pd.read_csv(DATA_PATH)

    # EDA từ notebook + kiểm tra mở rộng
    basic_eda(df)

    # Làm sạch
    df = clean_data(df)

    # Tách train / valid / test theo đúng tỷ lệ 80 / 10 / 10
    X_train, X_valid, X_test, y_train, y_valid, y_test = split_data(df)
    labels = sorted(df[TARGET_COL].unique())

    # Tuning hyperparameter trên train
    grid, tuning_results = tune_hyperparameters(
        X_train, y_train, n_iter=args.n_iter, n_jobs=args.n_jobs
    )
    best_model = grid.best_estimator_

    # Đánh giá trên train / valid / test
    train_metrics, train_report, train_cm = evaluate_split(
        best_model, X_train, y_train, 'train', labels
    )
    valid_metrics, valid_report, valid_cm = evaluate_split(
        best_model, X_valid, y_valid, 'valid', labels
    )
    test_metrics, test_report, test_cm = evaluate_split(
        best_model, X_test, y_test, 'test', labels
    )

    # Tổng hợp metrics
    summary_df = pd.DataFrame([train_metrics, valid_metrics, test_metrics])
    summary_df.to_csv(ARTIFACT_DIR / 'summary_metrics_all_splits.csv', index=False)

    print_header('BẢNG TỔNG HỢP METRICS')
    print(summary_df.to_string(index=False))

    # Kiểm tra overfitting
    compare_overfitting(train_metrics, valid_metrics, test_metrics, grid.best_score_)

    # Lưu model tốt nhất để dùng lại sau
    save_model(best_model, labels, grid.best_params_)
    save_inference_example()

    print_header('HOÀN THÀNH')
    print('Các file đầu ra đã được lưu trong thư mục artifacts_linear_svm/')

    # EDA từ notebook + kiểm tra mở rộng
    basic_eda(df)

    # Làm sạch
    df = clean_data(df)

    # Tách train / valid / test theo đúng tỷ lệ 80 / 10 / 10
    X_train, X_valid, X_test, y_train, y_valid, y_test = split_data(df)
    labels = sorted(df[TARGET_COL].unique())

    # Tuning hyperparameter trên train
    grid, tuning_results = tune_hyperparameters(X_train, y_train, n_iter=args.n_iter, n_jobs=args.n_jobs)
    best_model = grid.best_estimator_

    # Đánh giá trên train / valid / test
    train_metrics, train_report, train_cm = evaluate_split(best_model, X_train, y_train, 'train', labels)
    valid_metrics, valid_report, valid_cm = evaluate_split(best_model, X_valid, y_valid, 'valid', labels)
    test_metrics, test_report, test_cm = evaluate_split(best_model, X_test, y_test, 'test', labels)

    # Tổng hợp metrics
    summary_df = pd.DataFrame([train_metrics, valid_metrics, test_metrics])
    summary_df.to_csv(ARTIFACT_DIR / 'summary_metrics_all_splits.csv', index=False)

    print_header('BẢNG TỔNG HỢP METRICS')
    print(summary_df.to_string(index=False))

    # Kiểm tra overfitting
    compare_overfitting(train_metrics, valid_metrics, test_metrics, grid.best_score_)

    # Lưu model tốt nhất để dùng lại sau
    save_model(best_model, labels, grid.best_params_)
    save_inference_example()

    print_header('HOÀN THÀNH')
    print('Các file đầu ra đã được lưu trong thư mục artifacts_linear_svm/')


EDA TÓM TẮT
Shape dữ liệu: (23173, 10)

Các cột: ['review_id', 'Username', 'Address', 'Rating', 'Language', 'ID_clause', 'Clause', 'topic', 'code', 'topic_name']

Số lượng missing value theo cột:
review_id     0
Username      0
Address       0
Rating        0
Language      0
ID_clause     0
Clause        0
topic         0
code          0
topic_name    0
dtype: int64

Số dòng duplicated hoàn toàn: 0

Phân bố lớp (code):
code
VE     9017
TC     1872
LC     1734
RA     1697
HL     1477
CT     1475
SE     1411
UH     1102
TCr     984
CA      898
EL      698
EC      460
VR      348
Name: count, dtype: int64

Tỷ lệ lớp (%):
code
VE     38.91
TC      8.08
LC      7.48
RA      7.32
HL      6.37
CT      6.37
SE      6.09
UH      4.76
TCr     4.25
CA      3.88
EL      3.01
EC      1.99
VR      1.50
Name: proportion, dtype: float64

Độ dài Clause (ký tự) mô tả nhanh:
count    23173.00
mean        85.01
std         59.32
min          7.00
25%         47.00
50%         71.00
75%        104.00
max 